In [ ]:
"""F906: tail volume exhaustion reversal with 15-minute bars rebuilt from 1-minute data."""

import numpy as np
import pandas as pd
import dai


FACTOR_ID = "F906"
FACTOR_NAME = "tail_volume_exhaustion_reversal"


def main(datasources, start_date, end_date):
    bar1m = datasources["bar1m"]
    factorlib = "bigalpha_2026_factorlib"
    instruments = "bigalpha_2026_instruments"

    sql = f"""
    WITH minute_ranked AS (
        SELECT
            date AS ts,
            CAST(date AS DATE) AS trade_date,
            instrument,
            volume,
            ROW_NUMBER() OVER (
                PARTITION BY CAST(date AS DATE), instrument
                ORDER BY date ASC
            ) AS rn_day
        FROM {bar1m}
        WHERE CAST(date AS DATE) BETWEEN '{start_date}' AND '{end_date}'
    ),
    minute_bucketed AS (
        SELECT
            ts,
            trade_date,
            instrument,
            volume,
            CAST(FLOOR((rn_day - 1) / 15) AS BIGINT) AS bucket15
        FROM minute_ranked
    ),
    bar15_keys AS (
        SELECT
            trade_date,
            instrument,
            bucket15,
            MAX(ts) AS bar_ts
        FROM minute_bucketed
        GROUP BY trade_date, instrument, bucket15
    ),
    bar15m AS (
        SELECT
            m.trade_date,
            m.instrument,
            CAST(m.ts AS TIME) AS bar_time,
            m.volume
        FROM minute_bucketed m
        INNER JOIN bar15_keys k
          ON m.trade_date = k.trade_date
         AND m.instrument = k.instrument
         AND m.bucket15 = k.bucket15
         AND m.ts = k.bar_ts
    ),
    ordered AS (
        SELECT
            trade_date,
            instrument,
            bar_time,
            volume,
            LAG(volume) OVER (
                PARTITION BY trade_date, instrument
                ORDER BY bar_time ASC
            ) AS prev_volume
        FROM bar15m
    ),
    increments AS (
        SELECT
            trade_date,
            instrument,
            bar_time,
            CASE
                WHEN prev_volume IS NULL THEN volume
                WHEN volume >= prev_volume THEN volume - prev_volume
                ELSE 0
            END AS bar_volume
        FROM ordered
    ),
    volume_profile AS (
        SELECT
            trade_date AS date,
            instrument,
            SUM(CASE WHEN bar_time >= '14:30:00' THEN bar_volume ELSE 0 END) AS tail_volume,
            SUM(bar_volume) AS total_volume
        FROM increments
        GROUP BY trade_date, instrument
    ),
    daily AS (
        SELECT
            date,
            instrument,
            turn,
            change_ratio
        FROM {factorlib}
        WHERE date BETWEEN '{start_date}' AND '{end_date}'
    ),
    factor_raw AS (
        SELECT
            v.date,
            v.instrument,
            (
                COALESCE(v.tail_volume, 0) / NULLIF(v.total_volume, 0)
            ) * (
                -1.0 * COALESCE(d.change_ratio, 0)
            ) * (
                1.0 + COALESCE(d.turn, 0)
            ) / (
                1.0 + ABS(COALESCE(d.change_ratio, 0))
            ) AS factor
        FROM volume_profile v
        LEFT JOIN daily d
          ON v.date = d.date
         AND v.instrument = d.instrument
    )
    SELECT
        pool.date,
        pool.instrument,
        COALESCE(f.factor, 0.0) AS factor
    FROM {instruments} pool
    LEFT JOIN factor_raw f
      ON pool.date = f.date
     AND pool.instrument = f.instrument
    WHERE pool.date BETWEEN '{start_date}' AND '{end_date}'
    """

    result = dai.query(sql, filters={"date": [start_date, end_date]}).df()
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)
    result["factor"] = result["factor"].fillna(0.0)
    return result[["date", "instrument", "factor"]]


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时只构造平台会注入的基础数据源映射（逻辑名固定为 "bar1m"/"financial"）
    # 其他允许表在 main 内直接写物理表名，避免提交环境没有替换映射导致失败。
    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial'
    }
    # 本地用这段区间模拟「平台注入的测试集区间」（训练区间已在 main 内写死）
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 读取平台因子库用于回归评估，您可以换成自己的因子库
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统：
    # process_pools=False 表示不对因子库再做预处理（bigalpha_2026_factorlib 已处理过）
    # show=True 表示画出评估图表
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


In [ ]:
if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时只构造平台会注入的基础数据源映射（逻辑名固定为 "bar1m"/"financial"）
    # 其他允许表在 main 内直接写物理表名，避免提交环境没有替换映射导致失败。
    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial'
    }
    start_date = '2020-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"计算因子，全量测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    logger.info(f"读取因子库，全量测试区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    result_full = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
